### Context Window: Why Chatbots Forget

Every model has a context window limit - the maximum tokens it can process in a single request. 

This incudes input + output tokens combined.

Let's see what happens when you exceed it!

In [1]:
from google import genai
from google.genai import types
import os
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.environ['GOOGLE_API_KEY']
client = genai.Client(api_key=API_KEY)

In [2]:
# Create a VERY long message to hit the limit
long_story = 'Once upon a time there eas a programmer who loved python' * 100000

print(f"Story length: {len(long_story):,} characters")
print(f"Estimated tokens: ~{len(long_story) // 4:,} \n")

try:
    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=long_story,
        config={'max_output_tokens': 100}
    )
    print('Request succeeded!')
    print(f"Total Tokens used: {response.usage_metadata.total_token_count}:,")
    print(f"Gemini 2.5 flash has a HUGE context window (~1M tokens)")
    print(f" So this request fits comfortably!")
except Exception as e:
    print(f"ERROR: {e}")
    print("\n This is what happens when you exceed the context window!")


Story length: 5,600,000 characters
Estimated tokens: ~1,400,000 

ERROR: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 250000, model: gemini-2.5-flash\nPlease retry in 6.698645788s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_input_token_count', 'quotaId': 'GenerateContentInputTokensPerModelPerMinut

### What is Context Window?

Context Wndow = Maximum tokens in a single request
Different models have different limits:

- Gemini 2.5 Flash: ~1M tokens
- GPT-4: ~128K tokens
- Claude 4: ~200K tokens

### Context Window in Conversations


In chat applications, the context window includes:

- All previous messages (entire conversation history)
- Current message
- Response

As converdations grow, tokens accumulate!

In [3]:
messages = []

def chat(user_message):
    """Send message and track tokens"""
    messages.append(
        types.Content(role='user', 
        parts=[types.Part (text=user_message)])
    )
    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=messages
    )
    messages.append(
        types.Content(role='model', 
        parts=[types.Part (text=response.text)])
    )

    # Show token usage
    total_tokens = response.usage_metadata.total_token_count
    print(f"{response.text}")
    print(f"Total tokens used: {total_tokens} (include All) {len(messages)} messages\n")

    return response.text

# Start conversation 
print('Hi, My name is Anusha')
chat('Hi, My name is Anusha')

print('I am 21 years old')
chat('I am 21 years old')

print('I love python programming')
chat('I love python programming')

print('I work as AI Engineer')
chat('I work as AI Engineer')

print('What is my name')
chat('What is my name')

Hi, My name is Anusha
Hi Anusha! It's nice to meet you.

How can I help you today?
Total tokens used: 303 (include All) 2 messages

I am 21 years old
Okay, thanks for sharing that, Anusha. So you're 21 years old.

Is there something specific you wanted to talk about, or just sharing a bit about yourself? Either way is fine!
Total tokens used: 220 (include All) 4 messages

I love python programming
That's fantastic, Anusha! Python is a really popular and versatile language, and for good reason. It's used in so many exciting fields:

*   **Web development** (Django, Flask)
*   **Data science and machine learning** (NumPy, Pandas, scikit-learn, TensorFlow, PyTorch)
*   **Automation**
*   **Game development**
*   **Scientific computing**

What kind of Python programming do you enjoy most? Or what aspects of it are you most passionate about? I'm curious!
Total tokens used: 240 (include All) 6 messages

I work as AI Engineer
Ah, that makes perfect sense, Anusha! No wonder you love Python so 

'Your name is Anusha!'

In [4]:
messages = []

def chat(user_message):
    """Send message and track tokens"""
    messages.append(
        types.Content(role="user", parts=[types.Part(text=user_message)])
    )
    
    response = client.models.generate_content(
        model="gemini-2.5-flash-lite",
        contents=messages
    )
    
    messages.append(
        types.Content(role="model", parts=[types.Part(text=response.text)])
    )
    
    # Show token usage
    total_tokens = response.usage_metadata.total_token_count
    print(f"🤖: {response.text}")
    print(f"📊 Total tokens used: {total_tokens} (includes ALL {len(messages)} messages)\n")
    
    return response.text

# Start conversation 
print('Hi, My name is Anusha')
chat('Hi, My name is Anusha')

print('I am 21 years old')
chat('I am 21 years old')

print('I love python programming')
chat('I love python programming')

print('I work as AI Engineer')
chat('I work as AI Engineer')

print('What is my name')
chat('What is my name')

Hi, My name is Anusha


🤖: Hi Anusha, it's nice to meet you! How can I help you today?
📊 Total tokens used: 27 (includes ALL 2 messages)

I am 21 years old
🤖: Great to know, Anusha! Is there anything specific you'd like to discuss or ask about?
📊 Total tokens used: 57 (includes ALL 4 messages)

I love python programming
🤖: That's fantastic, Anusha! Python is a really versatile and popular programming language. What do you enjoy most about it?

Are you working on any particular projects, or are you looking to learn new things in Python? I'd love to hear more about your interests!
📊 Total tokens used: 120 (includes ALL 6 messages)

I work as AI Engineer
🤖: That's incredibly impressive, Anusha! Working as an AI Engineer at 21 is a significant accomplishment. Python is absolutely the language of choice for so many in the AI field.

What areas of AI are you most involved with? Are you focusing on:

*   **Machine Learning (ML)?**
*   **Deep Learning (DL)?**
*   **Natural Language Processing (NLP)?**
*   **Computer 

"Your name is Anusha! We've already established that. 😊"

### The Problem: Tokens Keep Growing

Notice how tokens increases with each message?

In real long conversation:
- Message 1: 50 tokens
- Message 10: 500 tokens
- Message 100: 2500 tokens
- Message 1000: 50,000 tokens

Eventually, you'll hit the context window limit!

### Solution: Remove old Messages

When approaching the limit, remove oldest messages

In [6]:
max_messages = 6 # keep only last 6 messages (3 exchanges)

def chat_with_limit(user_message):
    """Chat with context window management"""
    messages.append(
        types.Content(role='user',
        parts=[types.Part(text=user_message)])
    )

    # Remove old messages if too many
    if len(messages) > max_messages:
        removed = messages.pop(0) # Remove oldest
        print(f" Remove old messages: {removed.parts[0].text[:50]}...\n")

    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=messages
    )

    messages.append(
        types.Content(role='model',
        parts=[types.Part(text=response.text)]
        )
    )

    print(response.text)
    print(f"Messages in memory: {len(messages)}\n")

    return response.text

# Reset and try again
messages = []

print("My favorite color is blue")
chat_with_limit("My favorite color is blue")

print("I love tiramisu")
chat_with_limit("I love tiramisu")

print("I live in karachi")
chat_with_limit("I live in karachi")

print("I enjoy hiking")
chat_with_limit("I enjoy hiking")

# This will forget the first message!
print("What's my fvorite color")
chat_with_limit("What's my fvorite color")

My favorite color is blue
That's a wonderful choice! Blue is such a beautiful and versatile color. It's often associated with calmness, the sky, and the ocean.

Do you have a favorite shade of blue, like a deep navy, a vibrant sky blue, or something else?
Messages in memory: 2

I love tiramisu
Oh, excellent choice! Tiramisu is absolutely delicious. There's something so irresistible about that combination of coffee-soaked ladyfingers, rich mascarpone cream, and a dusting of cocoa.

It's one of those desserts that's just pure comfort and elegance all at once. What do you love most about it?
Messages in memory: 4

I live in karachi
Ah, Karachi! What a vibrant and bustling city. It's known for its incredible food scene, rich history, and being Pakistan's largest city.

I bet there are some amazing places to find tiramisu there! What do you enjoy most about living in Karachi?
Messages in memory: 6

I enjoy hiking
 Remove old messages: My favorite color is blue...

That's wonderful! Hiking i

"That's a fun question! But as an AI, I don't have personal information about you like your favorite color unless you share it with me.\n\nSo, I'm curious – what *is* your favorite color?"